# Impulse and Torque MLP (simple)

This notebook trains a physics-structured and physics informed MLP for a rigid body interacting with a plane-
As input we get the rotation as a 6-D rotation Matrix (with cos and sin)
We parameterise normal force with Hooke, similar to our simulations
We internally predict the contact normal
We couple torque to force via cross product: torque = r_{lever} x f
In the loss we use a Hube loss with a weight for energy conservation

In [1]:
%pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.2 MB/s eta 0:00:00a 0:00:01


In [2]:
%pip install trimesh
%pip install fast-simplification

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 12.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.6 MB/s eta 0:00:0000:010:01


In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import json
import math
from tqdm import tqdm
import trimesh

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print("Using device:", device)

Using device: cuda


## Constants

In [ ]:
TRAIN_TEST_SPLIT = 0.8

## Colab Only - Download data

In [6]:
def is_colab():
    try:
        import google.colab
        return True
    except Exception as e:
        return False

if is_colab():
    from google.colab import drive
    from tqdm import tqdm
    import os
    import json
    import shutil

    drive.mount('/content/drive', force_remount=True)

    # --- Copy the JSON file ---
    src_path = "/content/drive/MyDrive/final_output_contact_points.json"
    dst_path = "/content/final_output_contact_points.json"
    chunk_size = 1024 * 1024  # 1 MB
    file_size = os.path.getsize(src_path)

    with open(src_path, 'rb') as src, open(dst_path, 'wb') as dst:
        with tqdm(total=file_size, unit='B', unit_scale=True, desc="Copying JSON to /content") as pbar:
            while True:
                chunk = src.read(chunk_size)
                if not chunk:
                    break
                dst.write(chunk)
                pbar.update(len(chunk))
    print("Done! File is now in:", dst_path)

    # --- Copy the blender_models folder ---
    src_folder = "/content/drive/MyDrive/blender_models"
    dst_folder = "/content/blender_models"

    # Gather all files first so we know the total size for the progress bar
    all_files = []
    total_size = 0
    for root, dirs, files in os.walk(src_folder):
        for f in files:
            full = os.path.join(root, f)
            try:
                size = os.path.getsize(full)
            except OSError:
                size = 0
            all_files.append((full, size))
            total_size += size

    print(f"\nFound {len(all_files)} files in blender_models ({total_size / (1024**2):.1f} MB total)")

    os.makedirs(dst_folder, exist_ok=True)

    with tqdm(total=total_size, unit='B', unit_scale=True, desc="Copying blender_models") as pbar:
        for src_file, size in all_files:
            rel = os.path.relpath(src_file, src_folder)
            dst_file = os.path.join(dst_folder, rel)
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)

            with open(src_file, 'rb') as src, open(dst_file, 'wb') as dst:
                while True:
                    chunk = src.read(chunk_size)
                    if not chunk:
                        break
                    dst.write(chunk)
                    pbar.update(len(chunk))

    print("Done! Folder is now in:", dst_folder)

    # --- Print the first entry of the JSON ---
    with open(dst_path, 'r') as f:
        data = json.load(f)
    first = data[0] if isinstance(data, list) else next(iter(data.values()))
    print("\nFirst entry:")
    print(json.dumps(first, indent=2))

Mounted at /content/drive


Copying JSON to /content: 100%|██████████| 3.04G/3.04G [00:28<00:00, 107MB/s] 


Done! File is now in: /content/final_output_contact_points.json

Found 2 files in blender_models (13.1 MB total)


Copying blender_models: 100%|██████████| 13.7M/13.7M [00:02<00:00, 4.80MB/s]


Done! Folder is now in: /content/blender_models

First entry:
{
  "self_position": {
    "x": 2.912652891679045e-09,
    "y": 4.500506028285454e-09,
    "z": 0.30048527914508133
  },
  "linear_velocity": {
    "x": 4.411843891240104e-10,
    "y": 5.333639972857606e-10,
    "z": 0.048126627963949606
  },
  "angular_velocity": {
    "x": -0.0029046018813494202,
    "y": -0.00838692473483386,
    "z": -0.0017191488172748567
  },
  "self_rotation": {
    "qx": 0.17270058659965304,
    "qy": 0.9838035024726637,
    "qz": 0.037693681905239335,
    "qw": -0.02973822884911089,
    "roll": 3.077489925909082,
    "pitch": -0.07159373456854,
    "yaw": 2.79634261596079
  },
  "collider_position": {
    "x": 0.0,
    "y": 0.0,
    "z": 0.0
  },
  "collider_rotation": {
    "qx": 0.0,
    "qy": 0.0,
    "qz": 0.0,
    "qw": 1.0,
    "roll": 0.0,
    "pitch": -0.0,
    "yaw": 0.0
  },
  "relative_position_to_collider": {
    "x": 2.912652891679045e-09,
    "y": 4.500506028285454e-09,
    "z": 0.3004

## Dataset

Here we get the contat points with individual forces. From these forces, we calculate the torque and sum up the forces and torques to one force and one torque

In [7]:
class ContactDataset(Dataset):
    """World-frame wrench dataset for a rigid body on a plane.

    Features (13-D):
        [v_x, v_y, v_z,                           # linear velocity (world frame)
         w_x, w_y, w_z,                           # angular velocity (world frame)
         rel_pos_z,                               # height above plane
         sin(roll), cos(roll),
         sin(pitch), cos(pitch),
         sin(yaw), cos(yaw)]

    Per-sample extras (used by the GCN to place vertices in world space):
        self_position: (3,) world-frame position of the body origin (= COM
                       used in the torque computation: lever = r_world - cube_pos).

    Targets (world frame, PHYSICAL UNITS — not normalised):
        force:  (3,)   sum of per-contact forces
        torque: (3,)   sum of (r_world - com_world) x f_world
    """

    def __init__(self, data_list):
        features, forces, torques, collisions = [], [], [], []
        lin_vels, ang_vels, self_positions = [], [], []

        for contact in data_list:
            rel_pos = contact["relative_position_to_collider"]
            rel_rot = contact["relative_rotation_to_collider"]
            lin_vel = contact["linear_velocity"]
            ang_vel = contact["angular_velocity"]
            cube_pos = contact["self_position"]

            roll, pitch, yaw = rel_rot["roll"], rel_rot["pitch"], rel_rot["yaw"]
            feats = np.array([
                lin_vel["x"], lin_vel["y"], lin_vel["z"],
                ang_vel["x"], ang_vel["y"], ang_vel["z"],
                rel_pos["z"],
                np.sin(roll), np.cos(roll),
                np.sin(pitch), np.cos(pitch),
                np.sin(yaw), np.cos(yaw),
            ], dtype=np.float32)

            cube_pos_numpy = np.array([cube_pos["x"], cube_pos["y"], cube_pos["z"]],
                                      dtype=np.float32)
            force_numpy = np.zeros(3, dtype=np.float32)
            torque_numpy = np.zeros(3, dtype=np.float32)

            for p in contact.get("points", []):
                p_force = p["force"]
                lever_pos = p["contact_position_world"]
                point_force_numpy = np.array(
                    [p_force["x"], p_force["y"], p_force["z"]], dtype=np.float32)
                lever_rel_pos = np.array(
                    [lever_pos["x"], lever_pos["y"], lever_pos["z"]],
                    dtype=np.float32) - cube_pos_numpy

                force_numpy += point_force_numpy
                torque_numpy += np.cross(lever_rel_pos, point_force_numpy)

            features.append(feats)
            forces.append(force_numpy)
            torques.append(torque_numpy)
            collisions.append([min(len(contact.get("points", [])), 1)])
            lin_vels.append([lin_vel["x"], lin_vel["y"], lin_vel["z"]])
            ang_vels.append([ang_vel["x"], ang_vel["y"], ang_vel["z"]])
            self_positions.append(cube_pos_numpy)

        self.features       = torch.FloatTensor(np.asarray(features))
        self.forces         = torch.FloatTensor(np.asarray(forces))
        self.torques        = torch.FloatTensor(np.asarray(torques))
        self.collisions     = torch.FloatTensor(np.asarray(collisions))
        self.lin_vels       = torch.FloatTensor(np.asarray(lin_vels, dtype=np.float32))
        self.ang_vels       = torch.FloatTensor(np.asarray(ang_vels, dtype=np.float32))
        self.self_positions = torch.FloatTensor(np.asarray(self_positions, dtype=np.float32))

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return (
            self.features[idx],
            {
                "force":         self.forces[idx],
                "torque":        self.torques[idx],
                "is_collision":  self.collisions[idx],
                "lin_vel":       self.lin_vels[idx],
                "ang_vel":       self.ang_vels[idx],
                "self_position": self.self_positions[idx],
            },
        )

### Show dataset

show the first entry of the dataset

In [8]:
json_file = 'final_output_contact_points.json'
with open(json_file, 'r') as f:
    data = json.load(f)
if isinstance(data, dict):
    data = [data]

full_dataset = ContactDataset(data)


Print the first entry of the dataset

In [9]:
print(full_dataset[0])

(tensor([ 4.4118e-10,  5.3336e-10,  4.8127e-02, -2.9046e-03, -8.3869e-03,
        -1.7191e-03,  3.0049e-01,  6.4059e-02, -9.9795e-01, -7.1533e-02,
         9.9744e-01,  3.3843e-01, -9.4099e-01]), {'force': tensor([6.2812e-12, 4.7546e-12, 1.0848e+00]), 'torque': tensor([-1.3915e-02,  1.2797e-01, -2.7004e-13]), 'is_collision': tensor([1.]), 'lin_vel': tensor([4.4118e-10, 5.3336e-10, 4.8127e-02]), 'ang_vel': tensor([-0.0029, -0.0084, -0.0017]), 'self_position': tensor([2.9127e-09, 4.5005e-09, 3.0049e-01])})


Showing some info of the dataset

In [10]:
print("collisions:", int(full_dataset.collisions.sum().item()),
      "/", len(full_dataset))
mask = full_dataset.collisions.squeeze(-1).bool()
if mask.any():
    f_rms = full_dataset.forces[mask].pow(2).mean().sqrt().item()
    t_rms = full_dataset.torques[mask].pow(2).mean().sqrt().item()
    print(f"\nContact-only RMS force  = {f_rms:.4f}")
    print(f"Contact-only RMS torque = {t_rms:.4f}")
    print(f"Suggested w_torque / w_force ratio ≈ {f_rms / max(t_rms, 1e-8):.3f}")


collisions: 1116619 / 2145925

Contact-only RMS force  = 329.9835
Contact-only RMS torque = 82.1080
Suggested w_torque / w_force ratio ≈ 4.019


Split the dataset into train and test dataset and create data loaders

In [11]:
train_size = int(TRAIN_TEST_SPLIT * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator = torch.Generator())

use_gpu = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    num_workers=4 if use_gpu else 0,
    pin_memory=use_gpu,
    persistent_workers=use_gpu,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=4 if use_gpu else 0,
    pin_memory=use_gpu,
    persistent_workers=use_gpu,
)

Validation checks on the dataset:

In [12]:
input_dim = full_dataset.features.shape[1]
print(f"input_dim={input_dim}  (expect 10)")
print(f"force target shape = {tuple(full_dataset.forces.shape)}  (expect (N, 3))")
print(f"torque target shape = {tuple(full_dataset.torques.shape)}  (expect (N, 3))")
assert max(full_dataset.collisions) == 1


input_dim=13  (expect 10)
force target shape = (2145925, 3)  (expect (N, 3))
torque target shape = (2145925, 3)  (expect (N, 3))


## Mesh for GNN

In [13]:
import trimesh
import numpy as np

mesh = trimesh.load('blender_models/bunny.obj', 
                    process=True, force='mesh')
mesh.merge_vertices(merge_tex=True, merge_norm=True)

mesh = mesh.simplify_quadric_decimation(face_count=200)

print(mesh.vertices.shape)
print(len(mesh.vertex_neighbors))

l_vertex = []
l_connected = []
for i, neighbors in enumerate(mesh.vertex_neighbors):
    for neighbor in neighbors:
        l_vertex.append(i)
        l_connected.append(neighbor)

edge_index = np.array([l_vertex, l_connected])
vertice_positions = np.array(mesh.vertices)
num_vertices = len(mesh.vertices)

(102, 3)
102


## Model

Here is our physically structured model

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv, GlobalAttention
from torch_geometric.nn import global_max_pool

HEAD_OUT_DIM = 11  # 1 (collision) + 1 (depth) + 3 (force_residual) + 3 (normal) + 3 (lever)
VEL_SLICE = slice(0, 3)  # [vx, vy, vz] are the first 3 features


class ResBlock(nn.Module):
    def __init__(self, width, expansion=4):
        super().__init__()
        self.act = nn.ReLU(inplace=True)

        self.block = nn.Sequential(
            nn.LayerNorm(width),
            nn.Linear(width, width * expansion),
            self.act,
            nn.LayerNorm(width * expansion),
            nn.Linear(width * expansion, width),
            self.act,
        )

    def forward(self, x):
        return self.act(x + self.block(x))


class WrenchPredictor(nn.Module):
    """
    Predicts net wrench (force + torque) with a head whose force assembly
    matches the physics sim, plus a learned residual to absorb deviations
    the analytical model cannot express.

    Sim-matching part:
        F_spring  = -k * depth                        (depth <= 0)
        F_damping = -c * (v . n),  c = 2*sqrt(k*m)*b
        F_mag     = max(F_spring + F_damping, 0)      (hard clamp, no softplus)
        F_normal  = F_mag * n                         (along contact normal)

    Residual:
        F_vec     = F_normal + force_residual         (3-vec correction, unconstrained)
        T_vec     = lever x F_vec
    """

    def __init__(self, input_dim=13, width=256, num_blocks=5,
                 baseline_k=1e3, learn_k=True,
                 baseline_bounciness=0.5, learn_bounciness=True,
                 baseline_mass=1.0, learn_mass=True,
                 head_hidden=64):
        super().__init__()

        # --- backbone ---
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.LayerNorm(width),
            nn.GELU(),
        )

        backbone_layers = [nn.LayerNorm(width)]
        for _ in range(num_blocks, 1, -1):
            backbone_layers.append(ResBlock(width))
        self.backbone = nn.Sequential(*backbone_layers)

        self.head_trunk = nn.Sequential(
            nn.Linear(width, head_hidden),
            nn.LayerNorm(head_hidden),
            nn.GELU(),
        )
        self.head_out = nn.Linear(head_hidden, HEAD_OUT_DIM)

        # Physics parameters.
        self.k = nn.Parameter(
            torch.tensor(baseline_k, dtype=torch.float32),
            requires_grad=learn_k,
        )
        self.bounciness = nn.Parameter(
            torch.tensor(baseline_bounciness, dtype=torch.float32),
            requires_grad=learn_bounciness,
        )
        self.mass = nn.Parameter(
            torch.tensor(baseline_mass, dtype=torch.float32),
            requires_grad=learn_mass,
        )

    def forward(self, x):
        h = self.input_proj(x)
        h = self.backbone(h)

        raw = self.head_out(self.head_trunk(h))
        collision_logit, depth_raw, force_residual, normal_raw, lever = raw.split(
            [1, 1, 3, 3, 3], dim=-1
        )

        # Match sim: mass clamped to >= 1e-6, k positive.
        k_pos    = F.softplus(self.k)     if self.k.requires_grad else self.k
        mass_pos = F.softplus(self.mass)  if self.mass.requires_grad else self.mass
        mass_pos = torch.clamp(mass_pos, min=1e-6)
        bounciness = torch.sigmoid(self.bounciness)

        # Critical damping
        c = 2.0 * torch.sqrt(k_pos * mass_pos) * bounciness

        # Match sim's `min(penetration, 0.0)`: allow exact zero, clamp positives.
        depth = torch.clamp(depth_raw, max=0.0)

        # Normalize the contact normal (sim does the same defensively).
        contact_normal = F.normalize(normal_raw, dim=-1, eps=1e-8)

        F_spring = -k_pos * depth

        # Damping force along the normal.
        velocity = x[:, VEL_SLICE]
        vel_normal = (velocity * contact_normal).sum(dim=-1, keepdim=True)
        F_damping = -c * vel_normal

        # Sim uses a hard clamp at 0 — never sucks objects into surfaces.
        F_mag = F.relu(F_spring + F_damping)

        # Sim-matching normal force, plus learned residual correction.
        force_normal = F_mag * contact_normal
        force_vec    = force_normal + force_residual
        torque_vec   = torch.cross(lever, force_vec, dim=-1)

        return {
            "collision_logit": collision_logit,
            "force":  force_vec,
            "torque": torque_vec,
            "aux": {
                "contact_normal": contact_normal,
                "lever":          lever,
                "depth":          depth,
                "k":              k_pos.detach(),
                "c":              c.detach(),
                "mass":           mass_pos.detach(),
                "F_mag":          F_mag,
                "F_spring":       F_spring,
                "F_damping":      F_damping,
                "force_normal":   force_normal,
                "force_residual": force_residual,
            },
        }


# --------------------------------------------------------------------------
# World-space vertex transform
# --------------------------------------------------------------------------
# Feature layout (13-D, set by ContactDataset):
#   0:3   linear velocity     v_lin       (world frame)
#   3:6   angular velocity    omega       (world frame)
#   6     rel_pos_z           (height above plane)
#   7:9   (sin roll,  cos roll)
#   9:11  (sin pitch, cos pitch)
#   11:13 (sin yaw,   cos yaw)
#
# Rotation matrix is built directly from sin/cos pairs — no atan2 round-trip
# needed. Convention: intrinsic Z-Y-X (yaw, then pitch, then roll), i.e.
#     R = Rz(yaw) @ Ry(pitch) @ Rx(roll)
# which is what Blender exports for X-Y-Z Euler angles (the most common
# default). If the sim uses a different order, change `_rotmat_from_sincos`.
LIN_VEL_SLICE = slice(0, 3)
ANG_VEL_SLICE = slice(3, 6)
ROLL_SC_SLICE  = slice(7, 9)    # (sin, cos)
PITCH_SC_SLICE = slice(9, 11)
YAW_SC_SLICE   = slice(11, 13)


def _rotmat_from_sincos(features: torch.Tensor) -> torch.Tensor:
    """[B, F] feature batch -> [B, 3, 3] rotation matrix.

    Z-Y-X intrinsic: R = Rz(yaw) Ry(pitch) Rx(roll).
    """
    sr, cr = features[:, 7:8],  features[:, 8:9]
    sp, cp = features[:, 9:10], features[:, 10:11]
    sy, cy = features[:, 11:12], features[:, 12:13]

    # Each row is a [B, 3] tensor; stack along dim=1 -> [B, 3, 3].
    row0 = torch.cat([cy * cp,             cy * sp * sr - sy * cr,  cy * sp * cr + sy * sr], dim=-1)
    row1 = torch.cat([sy * cp,             sy * sp * sr + cy * cr,  sy * sp * cr - cy * sr], dim=-1)
    row2 = torch.cat([-sp,                 cp * sr,                 cp * cr               ], dim=-1)
    return torch.stack([row0, row1, row2], dim=1)


class GCNWrench(nn.Module):
    """
    Vertex features (per node, fed to conv1):
        local_xyz             3   (preserves shape identity)
        world_xyz             3   (vertex placed in world frame: R @ v_local + t)
        world_z               1   (explicit height — redundant with world_xyz
                                   but very useful for "is this vertex on/below
                                   the plane?")
        body_lin_vel          3
        body_ang_vel          3
        vertex_world_vel      3   (= v_lin + omega x r_world, the actual
                                   physical velocity at the vertex)
                                   ----
                                   total per-node geometric features = 16,
                                   plus broadcast `in_channels` features
                                   (the original 13-D per-sample state).
    """
    VERT_GEOM_DIM = 16  # local(3) + world(3) + world_z(1) + lin_vel(3) + ang_vel(3) + vel_at_vertex(3)

    def __init__(self, in_channels, hidden_channels, out_channels,
                 nodes_per_graph: int = num_vertices):
        super().__init__()
        # cached_=True so normalization is computed once; requires fixed edge_index
        self.conv1 = GCNConv(in_channels + self.VERT_GEOM_DIM, hidden_channels,
                             cached=True, add_self_loops=True, normalize=True)
        self.conv2 = GCNConv(hidden_channels, hidden_channels,
                             cached=True, add_self_loops=True, normalize=True)
        self.conv3 = GCNConv(hidden_channels, hidden_channels,
                             cached=True, add_self_loops=True, normalize=True)
        self.conv4 = GCNConv(hidden_channels, hidden_channels,
                             cached=True, add_self_loops=True, normalize=True)
        self.conv5 = GCNConv(hidden_channels, out_channels,
                             cached=True, add_self_loops=True, normalize=True)
        self.out_channels = out_channels
        self.N = nodes_per_graph
        self.head = WrenchPredictor(
            input_dim=in_channels + out_channels,
            width=256, num_blocks=5,
        )

    def forward(self, x, edge_index, vertex_pos, body_position=None):
        """
        Args:
            x:             [B*N, in_channels] per-node features (each sample's
                           state broadcast to all N nodes by `_broadcast_to_nodes`).
            edge_index:    [2, B*E] batched edge index.
            vertex_pos:    [N, 3] local-frame OBJ vertex positions (shared
                           across the batch — the rest pose).
            body_position: [B, 3] world-frame position of each body's origin
                           (the same point as `self_position` in the dataset,
                           which the torque target was computed about).
                           If None, defaults to zero translation (i.e. assume
                           the body is at the origin) — useful for benchmarks
                           where only the geometry matters.

        Returns:
            WrenchPredictor output dict.
        """
        N = self.N
        B = x.size(0) // N

        # batch vector: [0,0,...,0, 1,1,...,1, ..., B-1,...,B-1]
        batch = torch.arange(B, device=x.device).repeat_interleave(N)

        # --- Per-sample rotation matrix from the sin/cos features ---
        # x has been broadcast so x[k*N : (k+1)*N] are all identical and equal
        # to the sample-k state vector. Grab one row per sample.
        x_per_sample = x.view(B, N, -1)[:, 0, :]            # [B, in_channels]
        R = _rotmat_from_sincos(x_per_sample)               # [B, 3, 3]

        # --- Place local vertices in world space ---
        # vertex_pos: [N, 3] -> [B, N, 3]
        vp_local = vertex_pos.unsqueeze(0).expand(B, N, 3)  # [B, N, 3]

        # World rotation: (R @ v^T)^T == v @ R^T, batched per sample.
        # einsum: for each b, n: world_n_i = sum_j R[b, i, j] * vp_local[b, n, j]
        vp_world = torch.einsum("bij,bnj->bni", R, vp_local)  # [B, N, 3]

        if body_position is not None:
            vp_world = vp_world + body_position.unsqueeze(1)  # broadcast over N

        # --- Velocity at each vertex: v_at_vertex = v_lin + omega x r_world ---
        # r_world is the lever from the body's origin (the same anchor used in
        # the torque target). Use the rotated-but-not-translated vector here:
        # the lever is independent of where the body sits in space.
        v_lin   = x_per_sample[:, LIN_VEL_SLICE].unsqueeze(1)   # [B, 1, 3]
        omega   = x_per_sample[:, ANG_VEL_SLICE].unsqueeze(1)   # [B, 1, 3]
        # cross over the last dim, broadcasting omega across N
        r_world = torch.einsum("bij,bnj->bni", R, vp_local)     # [B, N, 3] (no translation)
        vel_at_vertex = v_lin + torch.cross(
            omega.expand_as(r_world), r_world, dim=-1)          # [B, N, 3]

        # Broadcast lin/ang vel onto every node.
        v_lin_n = v_lin.expand(B, N, 3)
        omega_n = omega.expand(B, N, 3)

        # --- Stack geometric features per node ---
        # Each piece is [B, N, *]; flatten the first two dims to [B*N, *].
        local_xyz_n   = vp_local.reshape(B * N, 3)
        world_xyz_n   = vp_world.reshape(B * N, 3)
        world_z_n     = vp_world[..., 2:3].reshape(B * N, 1)
        v_lin_n       = v_lin_n.reshape(B * N, 3)
        omega_n       = omega_n.reshape(B * N, 3)
        vel_at_vert_n = vel_at_vertex.reshape(B * N, 3)

        # Concatenate: original broadcast state + geometric per-vertex features.
        x_graph = torch.cat([
            x,                  # [B*N, in_channels]
            local_xyz_n,        # 3   shape identity
            world_xyz_n,        # 3   placed in world frame
            world_z_n,          # 1   explicit height above plane
            v_lin_n,            # 3
            omega_n,            # 3
            vel_at_vert_n,      # 3   rigid-body kinematics, ready-made
        ], dim=-1)

        h = F.relu(self.conv1(x_graph, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        h = F.relu(self.conv3(h, edge_index))
        h = F.relu(self.conv4(h, edge_index))
        h = self.conv5(h, edge_index)

        # Pass `size=B` so PyG's scatter doesn't call `int(index.max())`,
        # which forces a GPU->CPU sync and breaks CUDA graph capture.
        h_graph = global_max_pool(h, batch, size=B)          # [B, out_channels]
        x_linear = x.view(-1, N, x.size(-1)).mean(dim=1)     # [B, in_channels]

        fused = torch.cat([x_linear, h_graph], dim=-1)
        return self.head(fused)


def make_fast_predictor(input_dim=13, width=128, output_dim=32, num_blocks=5, baseline_k=1e3):
    model = GCNWrench(in_channels=input_dim, hidden_channels=width, out_channels=output_dim)
    return model

In [15]:
model = make_fast_predictor().to(device)

Print the model

In [16]:
print(f"Model: {model}")
print(f"Backbone: {model.head.backbone}")
print(f"Head trunk: {model.head.head_trunk}")
print(f"Head out: {model.head.head_out}")

Model: GCNWrench(
  (conv1): GCNConv(29, 128)
  (conv2): GCNConv(128, 128)
  (conv3): GCNConv(128, 128)
  (conv4): GCNConv(128, 128)
  (conv5): GCNConv(128, 32)
  (head): WrenchPredictor(
    (input_proj): Sequential(
      (0): Linear(in_features=45, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
    )
    (backbone): Sequential(
      (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (1): ResBlock(
        (act): ReLU(inplace=True)
        (block): Sequential(
          (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=256, out_features=1024, bias=True)
          (2): ReLU(inplace=True)
          (3): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (4): Linear(in_features=1024, out_features=256, bias=True)
          (5): ReLU(inplace=True)
        )
      )
      (2): ResBlock(
        (act): ReLU(inplace=True)
        (block): 

Check the model

In [17]:
model.eval()
with torch.no_grad():
    x = torch.zeros([1, 13]).expand(num_vertices, 13).to(device)
    model(x, edge_index=torch.tensor(edge_index, dtype=torch.long).to(device), vertex_pos=torch.tensor(vertice_positions, dtype=torch.float32).to(device))

### Benchmarking

Benchmark the evaluation speed of the model

In [18]:
NUM_BENCHMARKS = 1000
import time
import torch._logging
torch._logging.set_logs(recompiles=True, graph_breaks=True)

x = torch.rand((1, 13), device=device).expand(num_vertices, 13)
ei = torch.tensor(edge_index, dtype=torch.long, device=device)

results = []
benchmark_model = GCNWrench(13, 128, 30).to(device).eval()
benchmark_model = torch.compile(
    benchmark_model)
benchmark_model.eval()

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

    with torch.inference_mode():
        # warmup
        for _ in tqdm(range(1000), desc="Warmup"):
            benchmark_model(x, edge_index=ei, vertex_pos=torch.tensor(vertice_positions, device=device, dtype=torch.float32))
        print("=====Warmup completed======")

        for _ in tqdm(range(NUM_BENCHMARKS), desc="Benchmark"):
            torch.cuda.synchronize()
            start = time.perf_counter()
            benchmark_model(x, edge_index=ei, vertex_pos=torch.tensor(vertice_positions, device=device, dtype=torch.float32))
            torch.cuda.synchronize()
            end = time.perf_counter()
            results.append((end - start) * 1000)

    torch.backends.cudnn.benchmark = False
    per_pred_ms = sum(results) / NUM_BENCHMARKS
    print(f"Per prediction: {per_pred_ms:.2f} ms")

Warmup:   0%|          | 0/1000 [00:00<?, ?it/s]V0518 09:40:38.990000 8072 torch/_dynamo/symbolic_convert.py:4275] [0/0] [__graph_breaks] Graph break in user code at /usr/local/lib/python3.12/dist-packages/torch_geometric/utils/loop.py:650
V0518 09:40:38.990000 8072 torch/_dynamo/symbolic_convert.py:4275] [0/0] [__graph_breaks] Graph Break Reason: Encountered graph break when attempting to trace CALL: a function call, e.g. f(x, y):
V0518 09:40:38.990000 8072 torch/_dynamo/symbolic_convert.py:4275] [0/0] [__graph_breaks] 
V0518 09:40:38.990000 8072 torch/_dynamo/symbolic_convert.py:4275] [0/0] [__graph_breaks] Dynamic shape operator
V0518 09:40:38.990000 8072 torch/_dynamo/symbolic_convert.py:4275] [0/0] [__graph_breaks]   Explanation: Operator `aten.nonzero.default`'s output shape depends on input Tensor data.
V0518 09:40:38.990000 8072 torch/_dynamo/symbolic_convert.py:4275] [0/0] [__graph_breaks]   Hint: Enable tracing of dynamic shape operators with `torch._dynamo.config.capture_dyn

=====Warmup completed======


Benchmark: 100%|██████████| 1000/1000 [00:02<00:00, 396.93it/s]

Per prediction: 2.49 ms


In [19]:
if torch.cuda.is_available():
    print(f"\n=== Benchmark Results ({device}) ===")
    print(f"Max: {max(results)} ms")
    print(f"Min: {min(results)} ms")
    print(f"Avg: {sum(results) / len(results)} ms" )
    print(f"Median: {sorted(results)[len(results) // 2]} ms")
    print("Results:")
    print(results)


=== Benchmark Results (cuda) ===
Max: 4.115577999982634 ms
Min: 2.248828000119829 ms
Avg: 2.486738341997352 ms
Median: 2.4431269998785865 ms
Results:
[3.5881809999409597, 2.5956859999496373, 2.6992709999831277, 2.729977999933908, 2.46663100006117, 2.539029999752529, 2.473751999787055, 2.3343799998656323, 2.3720740000499063, 2.444138000100793, 2.440019999994547, 2.335674999812909, 2.4588080000285117, 2.4462480000693176, 2.912953999839374, 2.462540000124136, 2.5826920000326936, 2.5062319996322913, 2.3681500001657696, 2.402124999662192, 2.373375999923155, 2.433995000046707, 2.5003279997690697, 2.326366000033886, 2.35674800023844, 2.466085999913048, 2.4823300000207382, 2.4209640000663057, 2.379168000061327, 2.4347879998458666, 2.425088000109099, 2.3290730000553594, 2.3010400000202935, 2.4302679998982057, 2.4405550002484233, 2.3429799998666567, 2.479266000136704, 2.4630679999972926, 2.419387999907485, 2.4340810000467172, 3.1036250002216548, 2.467846999934409, 2.4973790000331064, 2.45345499

### CUDA Graph Benchmark

Captures the forward pass as a CUDA graph so the ~40 per-kernel dispatches are replaced by a single `cuGraphLaunch` on every call.

In [20]:
if torch.cuda.is_available():
    # Fixed batch size for the captured graph. If you need a different batch
    # size at inference, recapture (and rebuild edge_index with node offsets).
    B = 1
    N = num_vertices

    cuda_graph_model = GCNWrench(13, 128, 30).to(device).eval()

    # Static tensors — these addresses are baked into the captured graph.
    # Shape must exactly match what the model is replayed with later.
    static_input = torch.zeros(B * N, 13, device=device)
    static_edge_index = torch.tensor(edge_index, dtype=torch.long, device=device)
    static_vertex_pos = torch.tensor(vertice_positions, dtype=torch.float32, device=device)

    # --- Warmup before capture — must run on a side stream ---
    # Populates GCNConv's cached normalization, cuBLAS/cuDNN autotuning, etc.
    warmup_stream = torch.cuda.Stream()
    warmup_stream.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(warmup_stream):
        for _ in tqdm(range(1000), desc="CUDA graph warmup"):
            with torch.inference_mode():
                _ = cuda_graph_model(
                    static_input,
                    edge_index=static_edge_index,
                    vertex_pos=static_vertex_pos,
                )
    torch.cuda.current_stream().wait_stream(warmup_stream)
    torch.cuda.synchronize()

    # --- Capture ---
    cuda_graph = torch.cuda.CUDAGraph()
    with torch.inference_mode(), torch.cuda.graph(cuda_graph):
        static_output = cuda_graph_model(
            static_input,
            edge_index=static_edge_index,
            vertex_pos=static_vertex_pos,
        )
    print("CUDA graph captured.")

    # --- Benchmark ---
    # Pre-generate all inputs OUTSIDE the timing loop. `torch.rand` advances
    # the CUDA RNG offset; doing it inside the timed region adds noise and,
    # if a capture is ever in flight, can wedge the context.
    inputs = [
        torch.rand(B, 13, device=device)
            .unsqueeze(1).expand(B, N, 13).reshape(B * N, 13).contiguous()
        for _ in range(NUM_BENCHMARKS)
    ]
    torch.cuda.synchronize()

    cuda_graph_results = []
    with torch.inference_mode():
        for x in tqdm(inputs, desc="CUDA graph benchmark"):
            torch.cuda.synchronize()
            start = time.perf_counter()
            static_input.copy_(x)
            cuda_graph.replay()
            torch.cuda.synchronize()
            end = time.perf_counter()
            cuda_graph_results.append((end - start) * 1000)

    print(f"\n=== CUDA Graph Benchmark Results ({device}) ===")
    print(f"Max:    {max(cuda_graph_results):.4f} ms")
    print(f"Min:    {min(cuda_graph_results):.4f} ms")
    print(f"Avg:    {sum(cuda_graph_results) / len(cuda_graph_results):.4f} ms")
    print(f"Median: {sorted(cuda_graph_results)[len(cuda_graph_results) // 2]:.4f} ms")
    print(f"\nSpeedup over baseline: "
          f"{(sum(results) / len(results)) / (sum(cuda_graph_results) / len(cuda_graph_results)):.1f}x")

CUDA graph warmup: 100%|██████████| 1000/1000 [00:03<00:00, 301.62it/s]


CUDA graph captured.


CUDA graph benchmark: 100%|██████████| 1000/1000 [00:00<00:00, 1488.25it/s]


=== CUDA Graph Benchmark Results (cuda) ===
Max:    1.4888 ms
Min:    0.5028 ms
Avg:    0.6486 ms
Median: 0.5490 ms

Speedup over baseline: 3.8x


## 4. Loss

Huber (smooth-L1) loss replaces L1.  Near zero it's quadratic (smooth gradients,
won't over-punish tiny residuals); far from zero it's linear (robust to the
occasional outlier).  `delta=1.0` is in *normalized* target units so it's
roughly one standard deviation of the target.

**Energy-conservation penalty.**  For each collision sample we integrate one
timestep forward using the *predicted* wrench and check whether the body's
kinetic energy would grow beyond `e^2 * KE_before` (where `e` is the
coefficient of restitution).  Any excess is squared and added to the loss,
so the term is zero for physically admissible predictions and grows smoothly
when the model would inject energy — the exact failure mode you were seeing
at low collision speeds.  You control it with `w_energy`, `dt`, `mass`,
`inertia_diag`, and `restitution` when constructing `WrenchLoss`.


In [21]:
class WrenchLoss(nn.Module):
    """Physics-structured loss for a Hooke (linear elastic) contact model.

    Components:
        - BCE on collision_logit (with pos_weight for class imbalance).
        - Masked Huber on force and torque, operating in PHYSICAL units.
        - Soft penalty on f_n < 0 (Signorini violation).
        - Soft penalty on energy gain during collision (restitution-aware),
          using a BOUNDED log-based term so large violations at init don't
          produce enormous gradients that kill regression learning.

    Energy-conservation term
    ------------------------
    Given a predicted force F and torque tau applied over one timestep dt to a
    body with mass m and (diagonal) inertia I, the post-step velocities are
        v'   = v   + (F   / m) * dt
        w'   = w   + (I^-1 tau) * dt
    and the kinetic energy is  KE = 0.5 m |v|^2 + 0.5 w^T I w.
    For a real collision with coefficient of restitution e in [0, 1] we expect
        KE'  <=  e^2 * KE_before.
    We define the energy ratio
        r = KE' / (e^2 * KE_before)
    and penalise log(max(r, 1))^2. This is zero when r <= 1 (admissible),
    grows like (log r)^2 when r > 1, and its gradient in r is bounded — so a
    random-init network that predicts wildly wrong forces at epoch 0 won't
    produce an exploding energy gradient that pushes the model into the
    degenerate F≈0 basin.

    Warmup: w_energy typically starts at 0 and is ramped up over several
    epochs by the training loop via `set_energy_weight`, so the regression
    heads (force, torque) get to learn first before conservation pressure
    kicks in.

    Because targets are NOT pre-normalised, you may need to set w_force and
    w_torque to bring the two regression terms to comparable magnitude. A good
    heuristic is to set:
        w_force  ~ 1 / (force_rms_in_physical_units)
        w_torque ~ 1 / (torque_rms_in_physical_units)
    so both contribute roughly equally early in training.
    """
    def __init__(self,
                 w_force=1.0, w_torque=1.0, w_collision=1.0,
                 w_energy=0.0,
                 huber_delta=1.0, pos_weight=None,
                 dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                 restitution=0.5):
        super().__init__()
        self.w_force     = w_force
        self.w_torque    = w_torque
        self.w_collision = w_collision
        self.w_energy    = w_energy
        self.huber_delta = huber_delta
        self.dt          = float(dt)
        self.mass        = float(mass)
        self.restitution = float(restitution)
        # Inertia tensor (diagonal) for a unit cube by default: I = (1/6) m a^2
        # with m=1, a=1. Override via the constructor to match your simulated body.
        self.register_buffer(
            "inertia_diag",
            torch.tensor(inertia_diag, dtype=torch.float32),
        )
        # pos_weight is a tensor; register as buffer so .to(device) moves it.
        if pos_weight is not None and not torch.is_tensor(pos_weight):
            pos_weight = torch.tensor(float(pos_weight))
        self.register_buffer(
            "pos_weight",
            pos_weight if pos_weight is not None else torch.tensor(1.0),
        )
        self._has_pos_weight = pos_weight is not None

    def set_energy_weight(self, w):
        """Runtime hook for the training loop's warmup schedule."""
        self.w_energy = float(w)

    def _kinetic_energy(self, v, w):
        """KE = 0.5 m |v|^2 + 0.5 w^T I w for diagonal I. Shapes: (B,3)."""
        ke_lin = 0.5 * self.mass * (v * v).sum(dim=-1, keepdim=True)
        ke_rot = 0.5 * (self.inertia_diag * w * w).sum(dim=-1, keepdim=True)
        return ke_lin + ke_rot

    def forward(self, preds, targets):
        mask   = targets["is_collision"]                   # (B, 1)
        n_coll = mask.sum().clamp_min(1.0)

        # --- collision BCE ---
        loss_collision = F.binary_cross_entropy_with_logits(
            preds["collision_logit"], mask.float(),
            pos_weight=self.pos_weight if self._has_pos_weight else None,
            reduction="mean",
        )
        # --- masked Huber on force and torque (physical units) ---
        raw_f = F.huber_loss(preds["force"],  targets["force"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        raw_t = F.huber_loss(preds["torque"], targets["torque"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        loss_force  = (raw_f.sum(dim=-1, keepdim=True) * mask).sum() / n_coll
        loss_torque = (raw_t.sum(dim=-1, keepdim=True) * mask).sum() / n_coll

        # --- energy-conservation penalty (bounded, log-based) ---
        # Integrate one step using the predicted wrench and compare KE before vs after.
        # Only collision samples contribute (non-contact steps have F=tau=0 anyway).
        v = targets["lin_vel"]                              # (B, 3)
        w = targets["ang_vel"]                              # (B, 3)
        f_pred = preds["force"]                             # (B, 3)
        t_pred = preds["torque"]                            # (B, 3)

        v_next = v + (f_pred / self.mass) * self.dt
        # Diagonal inertia -> element-wise divide
        w_next = w + (t_pred / self.inertia_diag) * self.dt

        ke_before = self._kinetic_energy(v, w)              # (B, 1)
        ke_after  = self._kinetic_energy(v_next, w_next)    # (B, 1)

        # Log-ratio penalty:
        #   r = ke_after / (e^2 * ke_before + eps),  penalty = max(log r, 0)^2
        # Bounded gradient in F: d/dF log(ke_after) scales as 1/ke_after, so at
        # init where ke_after is huge the gradient is SMALL — the opposite of
        # (ke_after - budget)^2, which has gradient proportional to ke_after.
        eps       = 1e-6
        ke_budget = (self.restitution ** 2) * ke_before
        log_ratio = torch.log(ke_after + eps) - torch.log(ke_budget + eps)
        excess_log  = F.relu(log_ratio)                     # zero when admissible
        loss_energy = ((excess_log ** 2) * mask).sum() / n_coll

        total = (self.w_force     * loss_force
               + self.w_torque    * loss_torque
               + self.w_collision * loss_collision
               + self.w_energy    * loss_energy)

        # Diagnostic: fraction of collision samples that currently violate conservation,
        # plus the geometric-mean ratio so you can see HOW MUCH they violate by.
        with torch.no_grad():
            violating = ((excess_log > 0).float() * mask).sum() / n_coll
            # Mean log-ratio over collision samples (in log space so it's well-behaved)
            mean_log_ratio = (log_ratio * mask).sum() / n_coll

        return total, {
            "force":             loss_force.item(),
            "torque":            loss_torque.item(),
            "collision":         loss_collision.item(),
            "energy":            loss_energy.item(),
            "energy_violating":  violating.item(),
            "energy_mean_logr":  mean_log_ratio.item(),
            "w_energy":          self.w_energy,
            "total":             total.item(),
            "active_collisions": n_coll.item(),
            "k":                 preds["aux"]["k"].item(),
        }

## Training

In [22]:
def train_model(model, train_loader, val_loader, train_dataset,
                edge_index,  # <-- pass the graph in
                epochs=200, lr=1e-4, weight_decay=1e-4,
                w_force=1.0, w_torque=1.0, w_collision=0.5,
                # Energy-conservation warmup schedule.
                # Rationale: starting with w_energy>0 produces enormous gradients
                # at init (ke_after is huge for a random-init network) and pushes
                # the model into the degenerate F≈0 basin, where regression loss
                # plateaus at force_rms. Warming up from 0 lets the force/torque
                # heads learn a reasonable solution first; only then do we
                # gently tighten conservation.
                w_energy_max=0.1, energy_warmup_start=20, energy_warmup_epochs=30,
                dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                restitution=0.5):
    model.to(device)

    # Graph topology is constant across all samples — build once.
    ei = torch.as_tensor(edge_index, dtype=torch.long, device=device)
    NUM_NODES = num_vertices

    # Class-imbalance weight for BCE: #no-contact / #contact on train set.
    collisions = train_dataset.collisions.squeeze(-1).bool()
    n_pos = int(collisions.sum().item())
    n_neg = int((~collisions).sum().item())
    pos_weight = (n_neg / max(n_pos, 1)) if n_pos > 0 else 1.0
    print(f"BCE pos_weight = {pos_weight:.3f}  ({n_pos} contacts / {n_neg} non-contacts)")

    criterion = WrenchLoss(
        w_force=w_force, w_torque=w_torque, w_collision=w_collision,
        w_energy=0.0,  # ramped up by the warmup schedule below
        dt=dt, mass=mass, inertia_diag=inertia_diag, restitution=restitution,
        huber_delta=1.0, pos_weight=pos_weight,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=20
    )

    def energy_weight_at(epoch):
        """Linear ramp from 0 to w_energy_max over [start, start+epochs)."""
        if epoch < energy_warmup_start:
            return 0.0
        if energy_warmup_epochs <= 0:
            return float(w_energy_max)
        frac = (epoch - energy_warmup_start) / float(energy_warmup_epochs)
        return float(w_energy_max) * min(max(frac, 0.0), 1.0)

    best_val_loss = float('inf')
    loss_keys = ['total', 'force', 'torque', 'collision',
                 'energy', 'energy_violating', 'energy_mean_logr']

    for epoch in tqdm(range(epochs)):
        criterion.set_energy_weight(energy_weight_at(epoch))

        # --- Train ---
        model.train()
        train_losses = {k: 0.0 for k in loss_keys}
        for features, targets in train_loader:
            features = features.to(device)            # [B, 13]
            x = _broadcast_to_nodes(features, NUM_NODES)  # [B*8, 13]
            batch_ei = _batched_edge_index(ei, features.size(0), NUM_NODES)
            targets  = {k: v.to(device) for k, v in targets.items()}
            out = model(
                x,
                edge_index=batch_ei,
                vertex_pos=torch.tensor(vertice_positions, device=device, dtype=torch.float32),
                body_position=targets["self_position"],
            )
            optimizer.zero_grad()

            loss, components = criterion(out, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            for k in loss_keys:
                train_losses[k] += components[k]
        for k in loss_keys:
            train_losses[k] /= len(train_loader)

        # --- Validate ---
        model.eval()
        val_losses = {k: 0.0 for k in loss_keys}
        with torch.no_grad():
            for features, targets in val_loader:
                features = features.to(device)
                x = _broadcast_to_nodes(features, NUM_NODES)
                batch_ei = _batched_edge_index(ei, features.size(0), NUM_NODES)
                targets  = {k: v.to(device) for k, v in targets.items()}
                _, components = criterion(
                    model(
                        x,
                        edge_index=batch_ei,
                        vertex_pos=torch.tensor(vertice_positions, device=device, dtype=torch.float32),
                        body_position=targets["self_position"],
                    ),
                    targets,
                )
                for k in loss_keys:
                    val_losses[k] += components[k]
        for k in loss_keys:
            val_losses[k] /= len(val_loader)

        scheduler.step(val_losses['total'])

        if val_losses['total'] < best_val_loss:
            best_val_loss = val_losses['total']
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss':        best_val_loss,
            }, 'wrench_model_best.pth')

        if (epoch + 1) % 10 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            k_now  = components.get('k', float('nan'))
            we_now = criterion.w_energy
            print(f"Epoch {epoch+1}/{epochs} | LR: {lr_now:.2e} | k={k_now:.1f} | w_energy={we_now:.4f}")
            print(f"  Train: total={train_losses['total']:.4f} "
                  f"f={train_losses['force']:.4f} t={train_losses['torque']:.4f} "
                  f"c={train_losses['collision']:.4f} "
                  f"e={train_losses['energy']:.4f} vio={train_losses['energy_violating']:.2f} "
                  f"logr={train_losses['energy_mean_logr']:+.3f}")
            print(f"  Val:   total={val_losses['total']:.4f} "
                  f"f={val_losses['force']:.4f} t={val_losses['torque']:.4f} "
                  f"c={val_losses['collision']:.4f} "
                  f"e={val_losses['energy']:.4f} vio={val_losses['energy_violating']:.2f} "
                  f"logr={val_losses['energy_mean_logr']:+.3f}")

    return model


def _broadcast_to_nodes(features: torch.Tensor, num_nodes: int) -> torch.Tensor:
    """[B, F] -> [B*num_nodes, F] by repeating each sample's features for every node."""
    B, F = features.shape
    return features.unsqueeze(1).expand(B, num_nodes, F).reshape(B * num_nodes, F)


def _batched_edge_index(ei: torch.Tensor, batch_size: int, num_nodes: int) -> torch.Tensor:
    """
    Replicate edge_index for a batch, offsetting node indices per graph so that
    each sample's 8 nodes live in their own block. Result shape: [2, B*E].
    """
    E = ei.size(1)
    offsets = (torch.arange(batch_size, device=ei.device) * num_nodes).repeat_interleave(E)
    return ei.repeat(1, batch_size) + offsets.unsqueeze(0)

### Execute training

In [23]:
model = train_model(model, train_loader, val_loader, full_dataset,
                    epochs=200, lr=1e-3,
                    w_force=1.0, w_torque=1.0, w_collision=0.5, edge_index=edge_index)


BCE pos_weight = 0.922  (1116619 contacts / 1029306 non-contacts)


  0%|          | 0/200 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
  0%|          | 0/200 [00:45<?, ?it/s]


KeyboardInterrupt: 

## Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, loader, dataset, edge_index, device="cuda",
             rel_floor_force=0.05, rel_floor_torque=0.05):
    """Evaluate in physical units (targets were not normalised).

    Args:
        dataset: the *underlying* ContactDataset (not a Subset). Kept for API
                 symmetry; no target stats are needed since targets are physical.
        edge_index: graph topology (2, E) — replicated per batch sample.
        rel_floor_{force,torque}: targets with magnitude below this (in
                 physical units) are excluded from the relative-error stats
                 to avoid division-by-near-zero blow-up.
    """
    model.eval()
    all_abs_f, all_abs_t = [], []
    all_rel_f, all_rel_t = [], []
    all_coll_correct     = []

    # Physics-diagnostic accumulators (frictionless model: only f_n sign check)
    n_fn_neg = 0
    n_total  = 0

    # Graph topology is constant across all samples — build once.
    ei = torch.as_tensor(edge_index, dtype=torch.long, device=device)
    NUM_NODES = num_vertices

    for features, targets in loader:
        features = features.to(device)
        f_tgt = targets["force"].to(device)
        t_tgt = targets["torque"].to(device)
        c_tgt = targets["is_collision"].to(device).squeeze(-1).bool()

        x = _broadcast_to_nodes(features, NUM_NODES)
        batch_ei = _batched_edge_index(ei, features.size(0), NUM_NODES)
        preds = model(x, edge_index=batch_ei, vertex_pos=torch.tensor(vertice_positions, device=device, dtype=torch.float32))

        f_pred = preds["force"]
        t_pred = preds["torque"]
        c_pred = (torch.sigmoid(preds["collision_logit"]).squeeze(-1) > 0.5)

        all_coll_correct.append((c_pred == c_tgt).float().cpu())

        if c_tgt.any():
            f_pred_c = f_pred[c_tgt]
            f_tgt_c  = f_tgt[c_tgt]
            t_pred_c = t_pred[c_tgt]
            t_tgt_c  = t_tgt[c_tgt]

            # Targets are already physical — no de-normalisation needed.
            abs_f = (f_pred_c - f_tgt_c).norm(dim=-1)
            abs_t = (t_pred_c - t_tgt_c).norm(dim=-1)
            all_abs_f.append(abs_f.cpu())
            all_abs_t.append(abs_t.cpu())

            f_norm = f_tgt_c.norm(dim=-1)
            t_norm = t_tgt_c.norm(dim=-1)
            mask_f = f_norm > rel_floor_force
            mask_t = t_norm > rel_floor_torque
            if mask_f.any():
                rel_f = (f_pred_c[mask_f] - f_tgt_c[mask_f]).norm(dim=-1) / f_norm[mask_f]
                all_rel_f.append(rel_f.cpu())
            if mask_t.any():
                rel_t = (t_pred_c[mask_t] - t_tgt_c[mask_t]).norm(dim=-1) / t_norm[mask_t]
                all_rel_t.append(rel_t.cpu())

            # Physics diagnostic: how often the Hooke-plus-residual allows f_n < 0.
            f_n_c = preds["aux"]["force_residual"][c_tgt]
            n_fn_neg += int((f_n_c < 0).sum().item())
            n_total  += int(c_tgt.sum().item())

    abs_f = torch.cat(all_abs_f) if all_abs_f else torch.empty(0)
    abs_t = torch.cat(all_abs_t) if all_abs_t else torch.empty(0)
    rel_f = torch.cat(all_rel_f) if all_rel_f else torch.empty(0)
    rel_t = torch.cat(all_rel_t) if all_rel_t else torch.empty(0)
    coll_acc = torch.cat(all_coll_correct).mean().item()

    def fmt_pct(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.1%}  p90={e.quantile(0.9):.1%}  "
                f"p99={e.quantile(0.99):.1%}")
    def fmt_abs(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.4f}  p90={e.quantile(0.9):.4f}  "
                f"p99={e.quantile(0.99):.4f}")

    print("─" * 60)
    print(f"Collision accuracy : {coll_acc:.3%}")
    print(f"Force  abs err     : {fmt_abs(abs_f)}")
    print(f"Torque abs err     : {fmt_abs(abs_t)}")
    print(f"Force  rel err     : {fmt_pct(rel_f)}  "
          f"(on {rel_f.numel()} / {abs_f.numel()} samples above floor)")
    print(f"Torque rel err     : {fmt_pct(rel_t)}  "
          f"(on {rel_t.numel()} / {abs_t.numel()} samples above floor)")
    if n_total > 0:
        print(f"Physics violations : f_n<0 in {n_fn_neg}/{n_total} "
              f"({100*n_fn_neg/n_total:.2f}%)")
    print("─" * 60)

    return {
        "collision_acc":  coll_acc,
        "force_abs_p50":  abs_f.median().item() if abs_f.numel() else float("nan"),
        "force_abs_p99":  abs_f.quantile(0.99).item() if abs_f.numel() else float("nan"),
        "torque_abs_p50": abs_t.median().item() if abs_t.numel() else float("nan"),
        "torque_abs_p99": abs_t.quantile(0.99).item() if abs_t.numel() else float("nan"),
        "fn_neg_rate":    (n_fn_neg / n_total) if n_total > 0 else float("nan"),
    }

## Print 100 data points

In [ ]:
@torch.no_grad()
def print_predictions(model, loader, edge_index, n=100):
    """Print n predictions vs ground truth from the loader."""
    model.eval()
    all_f_pred, all_f_tgt = [], []
    all_t_pred, all_t_tgt = [], []
    all_coll_pred, all_coll_tgt = [], []

    ei = torch.as_tensor(edge_index, dtype=torch.long, device=device)
    NUM_NODES = num_vertices

    for features, targets in loader:
        features = features.to(device)
        x = _broadcast_to_nodes(features, NUM_NODES)
        batch_ei = _batched_edge_index(ei, features.size(0), NUM_NODES)
        body_position = targets["self_position"].to(device)
        preds = model(
            x,
            edge_index=batch_ei,
            vertex_pos=torch.tensor(vertice_positions, device=device, dtype=torch.float32),
            body_position=body_position,
        )

        all_f_pred.append(preds["force"].cpu())
        all_t_pred.append(preds["torque"].cpu())
        all_f_tgt.append(targets["force"])
        all_t_tgt.append(targets["torque"])
        all_coll_pred.append(torch.sigmoid(preds["collision_logit"]).cpu())
        all_coll_tgt.append(targets["is_collision"])
        collected = sum(x.shape[0] for x in all_f_pred)
        if collected >= n:
            break

    f_pred = torch.cat(all_f_pred)[:n]
    f_tgt  = torch.cat(all_f_tgt)[:n]
    t_pred = torch.cat(all_t_pred)[:n]
    t_tgt  = torch.cat(all_t_tgt)[:n]
    c_pred = torch.cat(all_coll_pred)[:n].squeeze(-1)
    c_tgt  = torch.cat(all_coll_tgt)[:n].squeeze(-1)

    header = (f"{'#':>4s}  {'coll':>5s} {'pred':>5s}  "
              f"{'force_pred':>30s}  {'force_true':>30s}  "
              f"{'torque_pred':>30s}  {'torque_true':>30s}  ")
    print(header)
    print("─" * len(header))
    for i in range(n):
        cp = f"{c_pred[i]:.2f}"
        ct = f"{int(c_tgt[i].item())}"
        fp = f"[{f_pred[i,0]:8.3f}, {f_pred[i,1]:8.3f}, {f_pred[i,2]:8.3f}]"
        ft = f"[{f_tgt[i,0]:8.3f}, {f_tgt[i,1]:8.3f}, {f_tgt[i,2]:8.3f}]"
        tp = f"[{t_pred[i,0]:8.4f}, {t_pred[i,1]:8.4f}, {t_pred[i,2]:8.4f}]"
        tt = f"[{t_tgt[i,0]:8.4f}, {t_tgt[i,1]:8.4f}, {t_tgt[i,2]:8.4f}]"
        print(f"{i:4d}  {ct:>5s} {cp:>5s}  {fp:>30s}  {ft:>30s}  {tp:>30s}  {tt:>30s}")


print_predictions(model, val_loader, edge_index, n=100)

   #   coll  pred                      force_pred                      force_true                     torque_pred                     torque_true  
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   0      1  0.99  [  -0.033,    0.018,  391.441]  [  -0.000,   -0.000,  383.223]  [140.7432, 217.6323,   0.0022]  [139.1342, 210.4139,   0.0000]
   1      1  0.99  [   0.003,    0.006,   67.368]  [   0.000,    0.000,   75.350]  [ -0.9098,   9.9806,  -0.0008]  [ -2.3514,   8.9866,  -0.0000]
   2      0  0.00  [   0.010,   -0.012,  165.242]  [   0.000,    0.000,    0.000]  [-28.6096,   4.2705,   0.0020]  [  0.0000,   0.0000,   0.0000]
   3      0  0.00  [   0.029,   -0.030,   97.187]  [   0.000,    0.000,    0.000]  [  0.7827,   0.8481,   0.0000]  [  0.0000,   0.0000,   0.0000]
   4      1  0.98  [  -0.006,   -0.004,  281.903]  [   0.000,    0.000,  288.013]  [-99.2935, -13.6189,  -0.0023]  [-101

Download checkpoint (Colab)

In [ ]:
if is_colab():
    from google.colab import files
    files.download("wrench_model_best.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>